In [1]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_ZIP = "/content/drive/MyDrive/gcp_pipeline.zip"
DATA_ZIPS = [
    "/content/drive/MyDrive/train_dataset-20260614T100421Z-3-001.zip",
    "/content/drive/MyDrive/train_dataset-20260614T100421Z-3-002.zip",
    "/content/drive/MyDrive/test_dataset-20260614T100423Z-3-001.zip",
]
PROJECT_DIR = "/content/gcp_pipeline"
DATA_DIR    = f"{PROJECT_DIR}/data"
TRAIN_DIR   = f"{DATA_DIR}/train_dataset"
TEST_DIR    = f"{DATA_DIR}/test_dataset"
CHECKPOINT_DIR   = "/content/drive/MyDrive/gcp_checkpoints"
PREDICTIONS_PATH = "/content/drive/MyDrive/predictions.json"


Mounted at /content/drive


In [2]:
import os, zipfile, shutil
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
with zipfile.ZipFile(PROJECT_ZIP, "r") as z:
    z.extractall("/content")
os.chdir(PROJECT_DIR)
print(os.getcwd())
print(os.listdir("."))

/content/gcp_pipeline
['.gitignore', 'requirements.txt', 'train.py', 'download_dataset.py', '__pycache__', 'dataset.py', 'infer.py', 'eda.py', 'utils.py', 'README.md', 'model.py']


In [3]:
!pip install -r requirements.txt

In [4]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [5]:
import os, zipfile
os.makedirs(DATA_DIR, exist_ok=True)
for zip_path in DATA_ZIPS:
    print("Extracting:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)

# Sanity check
print("Train exists:", os.path.isdir(TRAIN_DIR))
print("Test exists:", os.path.isdir(TEST_DIR))
# Filter out .DS_Store and __MACOSX directories for cleaner output
print("Train files:", len([f for f in os.listdir(TRAIN_DIR) if not (f.startswith('.') or f == '__MACOSX')]))
print("Test files:", len([f for f in os.listdir(TEST_DIR) if not (f.startswith('.') or f == '__MACOSX')]))

Extracting: /content/drive/MyDrive/train_dataset-20260614T100421Z-3-001.zip
Extracting: /content/drive/MyDrive/train_dataset-20260614T100421Z-3-002.zip
Extracting: /content/drive/MyDrive/test_dataset-20260614T100423Z-3-001.zip
Train exists: True
Test exists: True
Train files: 12
Test files: 73


In [6]:
!python train.py \
  --train_dir {TRAIN_DIR} \
  --output_dir {CHECKPOINT_DIR} \
  --backbone resnet34 \
  --epochs 40 \
  --batch_size 16 \
  --num_workers 2 \
  --resume auto

Using device: cuda
[build_samples] Skipped 4 malformed label entries.
Loaded 996 valid labeled samples.
Train samples: 874 | Val samples: 122
/content/gcp_pipeline/dataset.py:49: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(0.02, 0.11)),
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 139MB/s]
Class weights (['Cross', 'Square', 'L-Shaped']): [5.296969890594482, 0.8882113695144653, 0.593346893787384]
Epoch 1/40 | train_loss=1.8323 (kp=1.2009, cls=0.6314, px_err=129.35, acc=0.754) | val_loss=2.6430 (kp=1.2476, cls=1.3954, px_err=137.44, acc=0.254)
  -> Saved new best model to /content/drive/MyDrive/gcp_checkpoints/best_model.pt
  -> Saved latest checkpoint to /content/drive/MyDrive/gcp_checkpoints/last_checkpoint.pt
Epoch 2/40 | train_loss=1.3794 (kp=1.0100, cls=0.3693, px_err=108.93, acc=0.862) | val_loss=3.0167 (kp=

In [7]:
!python infer.py \
  --test_dir {TEST_DIR} \
  --checkpoint {CHECKPOINT_DIR}/best_model.pt \
  --output {PREDICTIONS_PATH} \
  --batch_size 32 \
  --num_workers 2

Using device: cuda
Loaded checkpoint trained with backbone=resnet34, epoch=12
Found 300 test images.
100% 10/10 [00:55<00:00,  5.53s/it]
Wrote predictions for 300 images to /content/drive/MyDrive/predictions.json


In [8]:
import json
import os

# Ensure the output directory exists for predictions.json
output_dir = os.path.dirname(PREDICTIONS_PATH)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# This cell should only run if predictions.json was successfully created by infer.py
if os.path.exists(PREDICTIONS_PATH):
    with open(PREDICTIONS_PATH) as f:
        preds = json.load(f)
    print("Total:", len(preds))
    for i, (k, v) in enumerate(preds.items()):
        print(k, v)
        if i == 4: break
else:
    print(f"Predictions file not found at {PREDICTIONS_PATH}. Please ensure infer.py ran successfully.")

Total: 300
231129_CTD/231129_CTD_GDA94/11/DJI_20231129142314_0151.JPG {'mark': {'x': 2104.39599609375, 'y': 1444.0589599609375}, 'verified_shape': 'Square'}
231129_CTD/231129_CTD_GDA94/2/DJI_20231129130143_0430.JPG {'mark': {'x': 2287.635009765625, 'y': 1462.6741943359375}, 'verified_shape': 'Square'}
231129_CTD/231129_CTD_GDA94/2/DJI_20231129130145_0431.JPG {'mark': {'x': 2301.70654296875, 'y': 1395.2523193359375}, 'verified_shape': 'Square'}
231129_CTD/231129_CTD_GDA94/21021511/DJI_20231129124916_0102.JPG {'mark': {'x': 2300.7431640625, 'y': 1451.83837890625}, 'verified_shape': 'Square'}
231129_CTD/231129_CTD_GDA94/22042006/DJI_20231129125223_0184.JPG {'mark': {'x': 2274.547607421875, 'y': 1433.53564453125}, 'verified_shape': 'Cross'}


In [9]:
from google.colab import files
import os

# Check if the file exists before attempting to download
if os.path.exists(PREDICTIONS_PATH):
    files.download(PREDICTIONS_PATH)
else:
    print(f"Predictions file not found at {PREDICTIONS_PATH}. Cannot download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>